# Olimpiadi estive 1964–2020: la crescita dello sport femminile nel medagliere

**Progetto finale — Master in Big Data and AI**

Questo notebook racconta, attraverso i dati, come è cambiato il peso dello sport femminile
alle Olimpiadi estive dal 1964 al 2020. L'obiettivo non è un'analisi statistica sofisticata,
ma una narrazione chiara e verificabile, costruita su tre domande di ricerca:

1. **Come sono cresciuti nel tempo la partecipazione e il peso delle medaglie femminili rispetto a quelli maschili?**
2. **Quali paesi costruiscono una parte rilevante del proprio medagliere sulle competizioni femminili?**
3. **I risultati maschili e femminili rispondono agli stessi fattori socio-economici?**

Dataset utilizzati:

- `data/csv_olimpiadi/Olympic_Athlete_Event_Details.csv` — una riga per atleta-evento: edizione, paese (NOC), sport, evento, medaglia;
- `data/csv_olimpiadi/Olympic_Athlete_Biography.csv` — anagrafica degli atleti, da cui ricaviamo il sesso;
- `src/santina/dataset_indicatori_definitivo.csv` — per ogni paese-edizione, la media degli indicatori
  socio-economici World Bank dei quattro anni precedenti l'edizione (già preparato: non viene ricostruito qui).

Tutti i grafici sono realizzati con **Altair**. Le conclusioni sono volutamente prudenti:
dove i dati non bastano a spiegare un fenomeno, la questione viene lasciata come domanda aperta.

In [1]:
from pathlib import Path
import re

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
alt.data_transformers.disable_max_rows()

# Palette coerente con gli altri notebook del progetto.
COLORE_M = "#3569a8"
COLORE_F = "#c7437a"
SCALA_GENERE = alt.Scale(domain=["Maschile", "Femminile"], range=[COLORE_M, COLORE_F])

## 2. Preparazione dei dati

I passaggi sono quattro, in ordine:

1. **caricamento** dei tre dataset (il notebook individua da solo la root del progetto);
2. **filtro** sulle sole Olimpiadi **estive** dal **1964 al 2020**;
3. **join** eventi ↔ biografie per recuperare il sesso di ogni atleta;
4. **pulizia**: classificazione del genere di ogni evento e deduplica delle medaglie di squadra.

La logica di aggregazione riprende quella già validata in `evoluzione_medagliere_definitivo.ipynb`.

In [2]:
def find_project_root() -> Path:
    """Risale le cartelle finché non trova la struttura dati del progetto."""
    expected = Path("data") / "csv_olimpiadi" / "Olympic_Athlete_Event_Details.csv"
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / expected).exists():
            return base
    raise FileNotFoundError("Non trovo la root del progetto con la cartella data/.")

PROJECT_ROOT = find_project_root()

events = pd.read_csv(
    PROJECT_ROOT / "data" / "csv_olimpiadi" / "Olympic_Athlete_Event_Details.csv",
    usecols=["edition", "country_noc", "sport", "event", "athlete_id", "medal", "isTeamSport"],
)
bio = pd.read_csv(
    PROJECT_ROOT / "data" / "csv_olimpiadi" / "Olympic_Athlete_Biography.csv",
    usecols=["athlete_id", "sex"],
)
socio = pd.read_csv(PROJECT_ROOT / "src" / "santina" / "dataset_indicatori_definitivo.csv", low_memory=False)

print(f"Righe atleta-evento: {len(events):,}")
print(f"Righe biografie: {len(bio):,}")
print(f"Righe indicatori socio-economici: {len(socio):,}")

Righe atleta-evento: 316,834
Righe biografie: 155,861
Righe indicatori socio-economici: 3,368


### Filtro temporale e join con le biografie

Teniamo solo le edizioni **estive dal 1964 al 2020** e agganciamo a ogni riga atleta-evento
il sesso dell'atleta. Il join copre il 100% delle righe: nessun atleta resta senza sesso.

In [3]:
# Filtro: solo Olimpiadi estive, anni 1964-2020.
events["year"] = events["edition"].str.extract(r"(\d{4})", expand=False).astype(int)
is_summer = events["edition"].str.contains("Summer Olympics", case=False, na=False)
events = events.loc[is_summer & events["year"].between(1964, 2020)].copy()

# Join con le biografie per il sesso dell'atleta.
events = events.merge(bio, on="athlete_id", how="left")
events["sesso_atleta"] = events["sex"].map({"Male": "Maschile", "Female": "Femminile"})

# Flag di comodo: la riga ha una medaglia? L'evento e' uno sport di squadra?
events["has_medal"] = events["medal"].notna() & events["medal"].astype("string").str.strip().ne("")
events["is_team_sport"] = (
    events["isTeamSport"] if events["isTeamSport"].dtype == bool
    else events["isTeamSport"].astype("string").str.strip().str.lower().eq("true")
)

print(f"Righe dopo il filtro Summer 1964-2020: {len(events):,}")
print(f"Righe senza sesso dopo il join: {events['sesso_atleta'].isna().sum()}")
print(f"Edizioni coperte: {sorted(events['year'].unique())}")

Righe dopo il filtro Summer 1964-2020: 179,141
Righe senza sesso dopo il join: 0
Edizioni coperte: [np.int64(1964), np.int64(1968), np.int64(1972), np.int64(1976), np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1996), np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020)]


### Genere degli eventi e controllo di qualità

Per il medagliere non usiamo il sesso dell'atleta ma il **genere dell'evento**, ricavato dal nome
(`100 metres, Women` → Femminile). Gli eventi **misti** (es. doppio misto, equitazione) restano in una
categoria separata e sono esclusi dalle quote di genere: attribuirli a un genere sarebbe arbitrario.

La tabella incrociata sotto è il controllo di qualità: gli eventi classificati "Femminile"
sono disputati quasi esclusivamente da atlete, e viceversa — la classificazione dal nome funziona.

In [4]:
def classifica_genere_evento(event: str) -> str:
    """Classifica un evento come Maschile, Femminile o Misto/Altro leggendone il nome."""
    text = str(event).lower()
    if re.search(r"\bwomen\b|women's|, women", text):
        return "Femminile"
    if re.search(r"\bmen\b|men's|, men", text):
        return "Maschile"
    return "Misto/Altro"

events["genere_evento"] = events["event"].apply(classifica_genere_evento)

display(pd.crosstab(events["genere_evento"], events["sesso_atleta"]))

sesso_atleta,Femminile,Maschile
genere_evento,,
Femminile,60312,3
Maschile,1,108599
Misto/Altro,1965,8261


### Deduplica delle medaglie di squadra

Il file atleta-evento riporta una riga per ogni componente della squadra: una vittoria della
pallavolo femminile compare 12 volte. Per contare le **medaglie** (non i medagliati):

- negli sport individuali ogni riga-medaglia vale una medaglia;
- negli sport di squadra teniamo una sola riga per `anno + paese + evento + tipo di medaglia`.

Come verifica, confrontiamo il totale ricostruito con il medagliere ufficiale presente nel
dataset degli indicatori: la correlazione è ≈0.99, con uno scarto medio inferiore a una
medaglia per paese-edizione — le piccole differenze dipendono da convenzioni di conteggio.

In [5]:
medal_rows = events.loc[events["has_medal"]]

team_medals = (
    medal_rows.loc[medal_rows["is_team_sport"]]
    .drop_duplicates(["year", "country_noc", "sport", "event", "medal"])
)
individual_medals = medal_rows.loc[~medal_rows["is_team_sport"]]
medal_units = pd.concat([individual_medals, team_medals], ignore_index=True)

# Medagliere per anno, paese e genere dell'evento (una colonna per genere).
medals_gender = (
    medal_units.groupby(["year", "country_noc", "genere_evento"], as_index=False)
    .size()
    .pivot_table(index=["year", "country_noc"], columns="genere_evento", values="size", fill_value=0)
    .reset_index()
    .rename(columns={"Femminile": "medaglie_f", "Maschile": "medaglie_m", "Misto/Altro": "medaglie_misto"})
)
medals_gender["medaglie_tot"] = medals_gender[["medaglie_f", "medaglie_m", "medaglie_misto"]].sum(axis=1)

# Verifica di coerenza con il medagliere ufficiale del dataset indicatori.
check = medals_gender.merge(
    socio[["Anno", "Codice_NOC", "Totale_Medaglie"]],
    left_on=["year", "country_noc"], right_on=["Anno", "Codice_NOC"], how="inner",
)
scarto = (check["medaglie_tot"] - check["Totale_Medaglie"]).abs()
print(f"Medaglie dopo la deduplica delle squadre: {len(medal_units):,}")
print(f"Coerenza con il medagliere ufficiale: r={check['medaglie_tot'].corr(check['Totale_Medaglie']):.4f}, "
      f"scarto medio assoluto={scarto.mean():.2f} medaglie per paese-edizione")

Medaglie dopo la deduplica delle squadre: 11,781
Coerenza con il medagliere ufficiale: r=0.9870, scarto medio assoluto=0.92 medaglie per paese-edizione


## 3. Evoluzione della partecipazione femminile

Contiamo gli **atleti unici** per edizione e sesso (un atleta iscritto a più gare conta una volta).
Il grafico composto mostra sopra i numeri assoluti, sotto la **quota femminile** sul totale dei
partecipanti, con la linea della parità al 50%.

In [6]:
def quota_femminile(df_long, col_valore):
    """Da un dataframe (year, genere, valore) calcola la quota femminile per anno."""
    wide = df_long.pivot_table(index="year", columns="genere", values=col_valore).reset_index()
    wide["quota_f"] = wide["Femminile"] / (wide["Femminile"] + wide["Maschile"])
    return wide[["year", "quota_f"]]

def grafico_serie_e_quota(df_long, col_valore, titolo_serie, titolo_quota):
    """Grafico composto: sopra le serie M/F nel tempo, sotto la quota femminile con la parita' al 50%."""
    linee = (
        alt.Chart(df_long)
        .mark_line(point=True, strokeWidth=2)
        .encode(
            x=alt.X("year:O", title=None),
            y=alt.Y(f"{col_valore}:Q", title=titolo_serie),
            color=alt.Color("genere:N", title="Genere", scale=SCALA_GENERE),
            tooltip=[
                alt.Tooltip("year:O", title="Anno"),
                alt.Tooltip("genere:N", title="Genere"),
                alt.Tooltip(f"{col_valore}:Q", title=titolo_serie, format=",.0f"),
            ],
        )
        .properties(width=720, height=280, title=titolo_serie)
    )

    quota = quota_femminile(df_long, col_valore)
    linea_quota = (
        alt.Chart(quota)
        .mark_line(point=True, strokeWidth=2.5, color=COLORE_F)
        .encode(
            x=alt.X("year:O", title="Anno"),
            y=alt.Y("quota_f:Q", title=titolo_quota, axis=alt.Axis(format="%"),
                    scale=alt.Scale(domain=[0, 0.6])),
            tooltip=[
                alt.Tooltip("year:O", title="Anno"),
                alt.Tooltip("quota_f:Q", title=titolo_quota, format=".1%"),
            ],
        )
    )
    parita = (
        alt.Chart(pd.DataFrame({"quota_f": [0.5]}))
        .mark_rule(strokeDash=[4, 4], color="#999999")
        .encode(y="quota_f:Q")
    )
    pannello_quota = (linea_quota + parita).properties(width=720, height=160, title=titolo_quota)

    return linee & pannello_quota

partecipanti = (
    events.dropna(subset=["sesso_atleta"])
    .drop_duplicates(subset=["year", "athlete_id"])
    .groupby(["year", "sesso_atleta"], as_index=False)
    .size()
    .rename(columns={"sesso_atleta": "genere", "size": "atleti"})
)

grafico_partecipazione = grafico_serie_e_quota(
    partecipanti, "atleti",
    "Atleti partecipanti per edizione (unici)",
    "Quota femminile dei partecipanti",
)
grafico_partecipazione

alt.VConcatChart(...)

**Lettura.** Nel 1964 le donne erano circa il 13% dei partecipanti; a Tokyo 2020 quasi il 48%.
La crescita è continua, senza inversioni, e accelera dagli anni '80 in poi. Il numero di atleti
uomini è sostanzialmente stabile dagli anni '90 (il CIO ha fissato un tetto ai partecipanti):
la crescita femminile è avvenuta soprattutto **ampliando il programma di gare femminili**,
non ingrandendo i Giochi. Londra 2012 è la prima edizione in cui le donne gareggiano in tutti
gli sport del programma.

## 4. Evoluzione del medagliere

Stessa struttura del grafico precedente, ma applicata alle **medaglie** (per genere dell'evento):
sopra i conteggi assoluti, sotto la quota femminile del medagliere. Gli eventi misti sono esclusi
dalla quota.

In [7]:
medaglie_per_anno = (
    medal_units.loc[medal_units["genere_evento"].isin(["Maschile", "Femminile"])]
    .groupby(["year", "genere_evento"], as_index=False)
    .size()
    .rename(columns={"genere_evento": "genere", "size": "medaglie"})
)

grafico_medagliere = grafico_serie_e_quota(
    medaglie_per_anno, "medaglie",
    "Medaglie assegnate per edizione, per genere dell'evento",
    "Quota femminile delle medaglie",
)
grafico_medagliere

alt.VConcatChart(...)

**Lettura.** La quota di medaglie femminili passa da circa il 21% del 1964 a oltre il 48% del 2020,
e la sua curva ricalca quella della partecipazione: quando il programma offre più gare femminili,
arrivano più medaglie femminili. È un punto importante per l'interpretazione: gran parte della
crescita del medagliere femminile è la crescita delle **opportunità di gara** decisa dal CIO,
più che un cambiamento nel rendimento delle atlete. La flessione visibile intorno al 1980-1984
riflette anche i boicottaggi (sezione 9).

## 5. Paesi che dipendono maggiormente dalle medaglie femminili

Per ogni paese aggreghiamo l'intero periodo 1964–2020 e calcoliamo la metrica più semplice
e leggibile:

$$\text{quota femminile} = \frac{\text{medaglie femminili}}{\text{medaglie femminili} + \text{medaglie maschili}}$$

Per evitare classifiche costruite su numeri piccoli (un paese con 2 medaglie su 3 femminili
finirebbe in testa), consideriamo solo i paesi con **almeno 30 medaglie di genere** nel periodo.

In [8]:
# Nome leggibile del paese: l'ultimo nome associato a ogni NOC nel dataset indicatori.
noc_nome = socio.sort_values("Anno").groupby("Codice_NOC")["Nazione"].last()

paesi = medals_gender.groupby("country_noc", as_index=False)[["medaglie_f", "medaglie_m"]].sum()
paesi["medaglie_fm"] = paesi["medaglie_f"] + paesi["medaglie_m"]
paesi["quota_f"] = paesi["medaglie_f"] / paesi["medaglie_fm"]
paesi["paese"] = paesi["country_noc"].map(noc_nome).fillna(paesi["country_noc"])

MIN_MEDAGLIE = 30
top15 = (
    paesi.loc[paesi["medaglie_fm"] >= MIN_MEDAGLIE]
    .sort_values("quota_f", ascending=False)
    .head(15)
)
print(f"Paesi con almeno {MIN_MEDAGLIE} medaglie di genere nel 1964-2020: "
      f"{(paesi['medaglie_fm'] >= MIN_MEDAGLIE).sum()}")

barre = (
    alt.Chart(top15)
    .mark_bar(color=COLORE_F)
    .encode(
        y=alt.Y("paese:N", sort="-x", title=None),
        x=alt.X("quota_f:Q", title="Quota femminile del medagliere 1964-2020",
                axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 0.7])),
        tooltip=[
            alt.Tooltip("paese:N", title="Paese"),
            alt.Tooltip("medaglie_f:Q", title="Medaglie femminili", format=",.0f"),
            alt.Tooltip("medaglie_m:Q", title="Medaglie maschili", format=",.0f"),
            alt.Tooltip("quota_f:Q", title="Quota femminile", format=".1%"),
        ],
    )
)
etichette = barre.mark_text(align="left", dx=4).encode(text=alt.Text("quota_f:Q", format=".0%"))
parita_v = (
    alt.Chart(pd.DataFrame({"quota_f": [0.5]}))
    .mark_rule(strokeDash=[4, 4], color="#555555")
    .encode(x="quota_f:Q")
)

grafico_top15 = (barre + etichette + parita_v).properties(
    width=680, height=380,
    title=f"Top 15 paesi per peso del medagliere femminile (min {MIN_MEDAGLIE} medaglie)",
)
grafico_top15

Paesi con almeno 30 medaglie di genere nel 1964-2020: 58


alt.LayerChart(...)

**Lettura.** I paesi oltre la linea del 50% hanno vinto **più medaglie con le donne che con gli
uomini** in oltre mezzo secolo di Giochi. Il gruppo di testa è eterogeneo: ci sono potenze dello
sprint come la Giamaica, paesi con sistemi sportivi statali che hanno investito presto sullo sport
femminile (Cina, Romania, ex DDR), e paesi occidentali come Paesi Bassi e Australia. Questa
eterogeneità è già un indizio per la domanda 3: non sembra esserci un unico profilo socio-economico
che produce un medagliere "femminile".

### Dove sono USA e Russia?

Un'assenza che colpisce: le due superpotenze del medagliere non compaiono nella Top 15. Il motivo
è la **metrica**: la classifica ordina per *quota* femminile, non per numero di medaglie, e le
grandi potenze storiche restano appena sotto la soglia di ingresso. La cella seguente mostra le
loro posizioni effettive nella classifica completa.

In [9]:
FOCUS_NOC = ["USA", "URS", "EUN", "RUS", "ROC"]

classifica = (
    paesi.loc[paesi["medaglie_fm"] >= MIN_MEDAGLIE]
    .sort_values("quota_f", ascending=False)
    .reset_index(drop=True)
)
classifica["posizione"] = classifica.index + 1

display(
    classifica.loc[classifica["country_noc"].isin(FOCUS_NOC),
                   ["posizione", "country_noc", "paese", "medaglie_m", "medaglie_f", "quota_f"]]
    .set_index("posizione")
)
print(f"Soglia di ingresso in Top 15 (15a posizione): "
      f"{classifica.loc[14, 'paese']} con quota femminile {classifica.loc[14, 'quota_f']:.1%}")

genere_evento,country_noc,paese,medaglie_m,medaglie_f,quota_f
posizione,,,,,
10,ROC,ROC,35.0,32.0,0.477612
16,RUS,Russian Federation,245.0,177.0,0.419431
17,USA,United States,832.0,584.0,0.412429
31,EUN,Unified Team,75.0,37.0,0.330357
40,URS,Soviet Union,507.0,191.0,0.273639


Soglia di ingresso in Top 15 (15a posizione): Austria con quota femminile 43.3%


Due fattori spiegano il risultato.

**1. L'effetto storico del programma gare.** USA e URSS vincono moltissimo fin dal 1964, quando
gli eventi femminili valevano solo il 21% delle medaglie: il loro enorme bottino degli anni
'60-'80 è strutturalmente sbilanciato sul maschile e abbassa la quota aggregata sull'intero
periodo. I paesi il cui successo è concentrato negli ultimi decenni — quando il programma è quasi
paritario — partono avvantaggiati in questa classifica. Non a caso, nel grafico della sezione
successiva (2000-2020) gli USA hanno *più* medaglie femminili che maschili.

**2. I NOC storici.** Il dataset segue il medagliere ufficiale del CIO, in cui ogni Comitato
Olimpico Nazionale (NOC) è un'entità distinta. La "Russia" è quindi spezzata su quattro sigle:

- **URS** — l'Unione Sovietica, fino a Seoul 1988 (include atleti di tutte le repubbliche);
- **EUN** — la *Squadra Unificata* (Équipe Unifiée) di Barcellona 1992, quando l'URSS si era
  appena dissolta e 12 ex repubbliche gareggiarono insieme sotto la bandiera olimpica;
- **RUS** — la Federazione Russa, da Atlanta 1996;
- **ROC** — il *Russian Olympic Committee* di Tokyo 2020, quando la Russia era squalificata come
  nazione per il doping di stato e i suoi atleti gareggiarono senza bandiera né inno.

Un pezzo di "Russia" in Top 15 in effetti c'è: ROC, che però rappresenta la sola edizione 2020.
Lo stesso vale per la Germania (GER unificata, FRG Ovest, GDR Est, EUA squadra unificata tedesca):
aggregare o meno questi NOC è una scelta analitica non banale, e qui manteniamo la convenzione
del medagliere ufficiale.

## 6. Il medagliere dei principali paesi, 2000–2020

Prima di passare agli indicatori socio-economici, guardiamo da vicino le potenze del medagliere
nel periodo recente (**2000–2020**, quando il programma femminile è ormai ampio e il confronto
tra generi è più equo). Riusiamo `medals_gender`, che contiene già le medaglie **deduplicate per
squadra**: basta filtrare gli anni e sommare per paese, senza ricaricare nulla.

Le barre mostrano, per i 15 paesi con più medaglie totali nel periodo, la scomposizione tra
medaglie maschili e femminili.

In [10]:
medaglie_recenti = medals_gender.loc[medals_gender["year"] >= 2000]

paesi_2000 = medaglie_recenti.groupby("country_noc", as_index=False)[
    ["medaglie_f", "medaglie_m", "medaglie_tot"]
].sum()
paesi_2000["quota_f"] = paesi_2000["medaglie_f"] / (paesi_2000["medaglie_f"] + paesi_2000["medaglie_m"])
paesi_2000["paese"] = paesi_2000["country_noc"].map(noc_nome).fillna(paesi_2000["country_noc"])

top15_2000 = paesi_2000.nlargest(15, "medaglie_tot")
top15_2000_long = top15_2000.melt(
    id_vars=["paese", "medaglie_tot", "quota_f"],
    value_vars=["medaglie_m", "medaglie_f"],
    var_name="genere", value_name="medaglie",
)
top15_2000_long["genere"] = top15_2000_long["genere"].map(
    {"medaglie_m": "Maschile", "medaglie_f": "Femminile"}
)

grafico_paesi_2000 = (
    alt.Chart(top15_2000_long)
    .mark_bar()
    .encode(
        y=alt.Y("paese:N", sort="-x", title=None),
        x=alt.X("medaglie:Q", title="Medaglie 2000-2020"),
        color=alt.Color("genere:N", title="Genere", scale=SCALA_GENERE),
        tooltip=[
            alt.Tooltip("paese:N", title="Paese"),
            alt.Tooltip("genere:N", title="Genere"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
            alt.Tooltip("quota_f:Q", title="Quota femminile", format=".1%"),
        ],
    )
    .properties(width=720, height=420,
                title="Medagliere maschile e femminile nei principali paesi, 2000-2020")
)
grafico_paesi_2000

alt.Chart(...)

**Lettura.** Tra le grandi potenze sportive l'equilibrio di genere non è affatto uniforme.
Gli Stati Uniti, primo medagliere del periodo, hanno vinto **più medaglie femminili che maschili**
(317 contro 300); Cina, Paesi Bassi e Canada superano nettamente il 50% femminile, e il Giappone
è esattamente alla pari. All'estremo opposto, paesi come Cuba, Italia e Francia costruiscono
ancora circa due terzi del medagliere sulle gare maschili. Essere una potenza olimpica, quindi,
non implica un medagliere equilibrato: la composizione di genere varia molto anche ai vertici
della classifica, e i dati a disposizione non bastano da soli a spiegarne il perché.

## 7. Relazione con gli indicatori socio-economici

Costruiamo un **panel paese-edizione** usando come base il dataset degli indicatori (3.368 righe):
così restano anche i paesi con zero medaglie, e le correlazioni non guardano solo i vincitori.
Ricordiamo che gli indicatori sono già la media dei quattro anni precedenti ogni edizione.

Due scelte metodologiche, semplici ma importanti:

- come misura di successo usiamo **log(1 + medaglie)**: la distribuzione delle medaglie è
  fortemente asimmetrica (pochi paesi vincono moltissimo) e il logaritmo evita che USA e URSS
  dominino ogni correlazione;
- per lo stesso motivo trasformiamo in logaritmo gli indicatori più asimmetrici (PIL, PIL pro
  capite, popolazione).

Usiamo solo gli indicatori con una copertura ragionevole (meno del 40% di valori mancanti);
le correlazioni sono **descrittive**: misurano associazioni, non causalità.

In [11]:
panel = (
    socio.rename(columns={"Anno": "year", "Codice_NOC": "country_noc", "Nazione": "paese"})
    .merge(medals_gender[["year", "country_noc", "medaglie_f", "medaglie_m"]],
           on=["year", "country_noc"], how="left")
)
panel[["medaglie_f", "medaglie_m"]] = panel[["medaglie_f", "medaglie_m"]].fillna(0)
panel["log_medaglie_f"] = np.log1p(panel["medaglie_f"])
panel["log_medaglie_m"] = np.log1p(panel["medaglie_m"])

panel["log_PIL_Assoluto"] = np.log1p(panel["PIL_Assoluto_USD"])
panel["log_PIL_Pro_Capite"] = np.log1p(panel["PIL_Pro_Capite_USD"])
panel["log_Popolazione"] = np.log1p(panel["Popolazione_Totale"])

INDICATORI = {
    "log_PIL_Assoluto": "log(PIL assoluto)",
    "log_PIL_Pro_Capite": "log(PIL pro capite)",
    "log_Popolazione": "log(Popolazione)",
    "Aspettativa_di_Vita": "Aspettativa di vita",
    "Tasso_Mortalita_Infantile": "Mortalita' infantile",
    "Tasso_Urbanizzazione_perc": "Urbanizzazione %",
    "Popolazione_Eta_Lavorativa_perc": "Popolazione in eta' lavorativa %",
    "Crescita_PIL_perc_annua": "Crescita del PIL %",
}
for col in INDICATORI:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

print(f"Osservazioni paese-edizione: {len(panel):,} | paesi distinti: {panel['country_noc'].nunique()}")

Osservazioni paese-edizione: 3,368 | paesi distinti: 218


In [12]:
# Correlazione di Pearson di ogni indicatore con log(1+medaglie), separatamente per genere.
records = []
for col, label in INDICATORI.items():
    for target, genere in [("log_medaglie_m", "Maschile"), ("log_medaglie_f", "Femminile")]:
        tmp = panel[[col, target]].replace([np.inf, -np.inf], np.nan).dropna()
        records.append({"indicatore": label, "genere": genere,
                        "r": tmp[col].corr(tmp[target]), "n": len(tmp)})
corr_genere = pd.DataFrame(records)

ordine = (
    corr_genere.loc[corr_genere["genere"] == "Femminile"]
    .sort_values("r", ascending=False)["indicatore"].tolist()
)

punti = (
    alt.Chart(corr_genere)
    .mark_circle(size=120, opacity=0.9, stroke="#333333", strokeWidth=0.4)
    .encode(
        y=alt.Y("indicatore:N", sort=ordine, title=None),
        x=alt.X("r:Q", title="Correlazione con log(1 + medaglie)",
                scale=alt.Scale(domain=[-0.4, 0.7])),
        color=alt.Color("genere:N", title="Medagliere", scale=SCALA_GENERE),
        tooltip=[
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("genere:N", title="Medagliere"),
            alt.Tooltip("r:Q", title="r di Pearson", format=".3f"),
            alt.Tooltip("n:Q", title="Osservazioni", format=",.0f"),
        ],
    )
)
zero = alt.Chart(pd.DataFrame({"r": [0.0]})).mark_rule(color="#999999").encode(x="r:Q")

grafico_correlazioni = (zero + punti).properties(
    width=680, height=320,
    title="Gli stessi fattori? Correlazione degli indicatori con il medagliere, per genere",
)
grafico_correlazioni

alt.LayerChart(...)

**Lettura.** Gli indicatori di **dimensione** — PIL assoluto e popolazione — sono i più associati
al numero di medaglie: i paesi grandi e ricchi vincono di più, e questo vale per entrambi i generi.
Il PIL pro capite (ricchezza media) conta, ma meno della dimensione complessiva. Gli indicatori di
sviluppo sociale (aspettativa di vita, urbanizzazione) mostrano associazioni deboli e la mortalità
infantile una debole associazione negativa. La crescita del PIL è sostanzialmente scorrelata:
vince chi è grande, non chi cresce in fretta. Nessun indicatore supera r≈0.6: gli indicatori
socio-economici generali spiegano solo in parte il successo olimpico.

## 8. Confronto uomini vs donne

Il dot plot della sezione precedente contiene già la risposta visiva: per ogni indicatore, il punto
blu (maschile) e quello rosa (femminile) sono **quasi sovrapposti**. La tabella sotto quantifica le
differenze, poi guardiamo la relazione più forte — la ricchezza — con uno scatter e due rette di
regressione separate. Ci limitiamo al periodo **2000–2020**, quando il programma femminile è ormai
ampio e il confronto tra generi è più equo.

In [13]:
confronto = (
    corr_genere.pivot_table(index="indicatore", columns="genere", values="r")
    .reindex(ordine)
    .assign(differenza=lambda d: d["Femminile"] - d["Maschile"])
    .round(3)
)
display(confronto)
print(f"Differenza assoluta media tra i generi: {confronto['differenza'].abs().mean():.3f}")

genere,Femminile,Maschile,differenza
indicatore,,,
log(PIL assoluto),0.581,0.613,-0.032
log(Popolazione),0.391,0.451,-0.060
Popolazione in eta' lavorativa %,0.366,0.382,-0.016
log(PIL pro capite),0.351,0.327,0.024
Aspettativa di vita,0.336,0.348,-0.012
Urbanizzazione %,0.274,0.292,-0.017
Crescita del PIL %,-0.064,-0.060,-0.004
Mortalita' infantile,-0.332,-0.352,0.020


Differenza assoluta media tra i generi: 0.023


In [14]:
recente = panel.loc[panel["year"] >= 2000]

scatter_data = (
    recente[["year", "paese", "log_PIL_Pro_Capite", "medaglie_m", "medaglie_f"]]
    .melt(id_vars=["year", "paese", "log_PIL_Pro_Capite"],
          value_vars=["medaglie_m", "medaglie_f"], var_name="genere", value_name="medaglie")
    .dropna(subset=["log_PIL_Pro_Capite"])
)
scatter_data["genere"] = scatter_data["genere"].map({"medaglie_m": "Maschile", "medaglie_f": "Femminile"})
scatter_data["log_medaglie"] = np.log1p(scatter_data["medaglie"])

base = alt.Chart(scatter_data)
punti_scatter = base.mark_circle(opacity=0.25, size=40).encode(
    x=alt.X("log_PIL_Pro_Capite:Q", title="log(PIL pro capite)", scale=alt.Scale(zero=False)),
    y=alt.Y("log_medaglie:Q", title="log(1 + medaglie)"),
    color=alt.Color("genere:N", title="Medagliere", scale=SCALA_GENERE),
    tooltip=[
        alt.Tooltip("paese:N", title="Paese"),
        alt.Tooltip("year:O", title="Anno"),
        alt.Tooltip("genere:N", title="Genere"),
        alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
    ],
)
rette = (
    base.transform_regression("log_PIL_Pro_Capite", "log_medaglie", groupby=["genere"])
    .mark_line(strokeWidth=3)
    .encode(x="log_PIL_Pro_Capite:Q", y="log_medaglie:Q",
            color=alt.Color("genere:N", scale=SCALA_GENERE))
)

grafico_pil = (punti_scatter + rette).properties(
    width=720, height=400,
    title="PIL pro capite e medaglie per genere, 2000-2020",
).interactive()
grafico_pil

alt.LayerChart(...)

**Lettura.** Le due rette di regressione sono quasi parallele e quasi sovrapposte: a parità di
ricchezza, un paese tende a vincere un numero simile di medaglie maschili e femminili. Anche nella
tabella delle correlazioni le differenze tra i generi sono di pochi centesimi, piccole rispetto
alla forza dei fattori comuni.

La risposta qualitativa alla domanda 3 è quindi: **sì, i medaglieri maschile e femminile rispondono
sostanzialmente agli stessi fattori socio-economici** — dimensione economica e demografica prima di
tutto. Ciò che gli indicatori *non* spiegano è la variabilità tra paesi simili vista nella sezione 5:
lì contano probabilmente politiche sportive, tradizioni e fattori culturali che non sono misurati
dal World Bank.

## 9. Boicottaggi olimpici: una perturbazione da tenere presente

Tre edizioni del nostro periodo sono state segnate da boicottaggi di massa. Il numero esatto di
atleti esclusi non è documentato in modo uniforme, quindi riportiamo il **numero di paesi**
coinvolti (valori stimati, arrotondati, dalle fonti storiche del CIO).

In [15]:
boicottaggi = pd.DataFrame([
    {
        "Anno": 1976, "Edizione": "Montreal 1976",
        "Motivo": "Boicottaggio africano: protesta contro la presenza della Nuova Zelanda, "
                  "il cui rugby aveva giocato nel Sudafrica dell'apartheid",
        "Paesi coinvolti (stima)": "~30 (paesi africani, piu' Iraq e Guyana)",
    },
    {
        "Anno": 1980, "Edizione": "Mosca 1980",
        "Motivo": "Boicottaggio guidato dagli USA in risposta all'invasione sovietica "
                  "dell'Afghanistan",
        "Paesi coinvolti (stima)": "~60-65 (tra cui USA, Germania Ovest, Giappone, Canada, Cina)",
    },
    {
        "Anno": 1984, "Edizione": "Los Angeles 1984",
        "Motivo": "Boicottaggio del blocco sovietico, ufficialmente per motivi di sicurezza, "
                  "di fatto in risposta al 1980",
        "Paesi coinvolti (stima)": "14 (tra cui URS, DDR, Cuba)",
    },
])
display(boicottaggi.set_index("Anno"))

,Edizione,Motivo,Paesi coinvolti (stima)
Anno,,,
1976,Montreal 1976,Boicottaggio africano: protesta contro la pres...,"~30 (paesi africani, piu' Iraq e Guyana)"
1980,Mosca 1980,Boicottaggio guidato dagli USA in risposta all...,"~60-65 (tra cui USA, Germania Ovest, Giappone,..."
1984,Los Angeles 1984,"Boicottaggio del blocco sovietico, ufficialmen...","14 (tra cui URS, DDR, Cuba)"


**Perché conta per questa analisi.** I boicottaggi perturbano proprio le serie che abbiamo
costruito: nel 1980 e nel 1984 mancano dal medagliere blocchi interi di paesi, e alcuni di questi
(URSS, DDR) erano tra i più forti proprio nello sport femminile. La quota femminile di medaglie
di quelle edizioni — e le correlazioni con gli indicatori in quegli anni — vanno quindi lette con
cautela: parte delle oscillazioni intorno al 1980-1984 non riflette tendenze reali ma l'assenza
forzata di alcuni paesi. È uno dei motivi per cui, nel confronto tra generi, abbiamo privilegiato
il periodo 2000-2020.

## 10. Conclusioni

**Domanda 1 — Come sono cresciuti partecipazione e peso delle medaglie femminili?**

- *Cosa emerge chiaramente*: una crescita continua e senza inversioni. La quota femminile dei
  partecipanti passa da ~13% (1964) a ~48% (2020); la quota di medaglie femminili da ~21% a ~48%.
  Tokyo 2020 è di fatto la prima edizione quasi paritaria.
- *Cosa suggerisce l'analisi*: le due curve viaggiano insieme — la crescita del medagliere
  femminile è soprattutto la crescita delle opportunità di gara (più eventi femminili in
  programma), non un cambiamento di rendimento.
- *Limiti*: non abbiamo normalizzato per il numero di eventi in programma per genere; il genere
  dell'evento è ricavato dal nome dell'evento.
- *Domanda aperta*: quanta parte della crescita è "meccanica" (decisioni del CIO sul programma)
  e quanta riflette investimenti reali dei paesi nello sport femminile?

**Domanda 2 — Quali paesi costruiscono il medagliere sulle donne?**

- *Cosa emerge chiaramente*: esiste un gruppo di paesi che ha vinto più con le donne che con gli
  uomini (quota femminile > 50% su tutto il periodo), e il gruppo è eterogeneo: Giamaica, Cina,
  Romania, Paesi Bassi, Canada hanno storie sportive molto diverse.
- *Cosa suggerisce l'analisi*: non sembra esserci un unico profilo (economico o geografico) dei
  paesi "al femminile"; pesano probabilmente scelte specifiche di investimento sportivo.
- *Limiti*: la soglia minima di medaglie (30) e l'aggregazione su 56 anni sono scelte nostre;
  soglie o periodi diversi cambierebbero in parte la classifica. L'aggregazione sull'intero
  periodo penalizza inoltre le potenze storiche (USA, URSS), forti già quando il programma
  femminile era piccolo, e i paesi che hanno cambiato assetto politico compaiono sotto più NOC
  (URS/EUN/RUS/ROC; GER/FRG/GDR/EUA), come nel medagliere ufficiale.
- *Domanda aperta*: perché paesi socio-economicamente simili hanno quote femminili così diverse?
  Servirebbero dati su politiche sportive e investimenti federali, che non abbiamo.

**Domanda 3 — Stessi fattori socio-economici per uomini e donne?**

- *Cosa emerge chiaramente*: le correlazioni con gli indicatori World Bank sono quasi identiche
  per i due generi (differenze di pochi centesimi). Dimensione economica (PIL) e demografica
  (popolazione) sono i fattori più associati al successo per entrambi.
- *Cosa suggerisce l'analisi*: il successo olimpico femminile non sembra richiedere condizioni
  socio-economiche diverse da quello maschile; le differenze tra paesi nella composizione di
  genere del medagliere restano in gran parte non spiegate da questi indicatori.
- *Limiti*: correlazioni descrittive, non causali; ~20-40% di valori mancanti su alcuni
  indicatori; i boicottaggi perturbano le edizioni 1976-1984; nessun controllo multivariato.
- *Domanda aperta*: un modello con effetti fissi per paese, o dati su spesa sportiva e
  partecipazione femminile allo sport di base, potrebbero spiegare la parte che qui resta
  inspiegata. Con i soli indicatori generali del World Bank, non è possibile.

**Nota metodologica finale.** Tutte le scelte (deduplica squadre, esclusione eventi misti,
soglie minime, trasformazioni log) sono dichiarate nel notebook e facilmente modificabili:
l'obiettivo era la trasparenza del racconto, non la massima sofisticazione statistica.

## 11. Esportazione dei grafici Altair

Questa cella esporta i sei grafici del notebook in formato JSON Vega-Lite dentro
`src/stefano/charts_def2`, richiamabili dal sito con il tag `vegachart`:

```liquid
{% vegachart /assets/charts/01_partecipazione_e_quota.json %}
```

In [16]:
CHARTS_DIR = PROJECT_ROOT / "src" / "stefano" / "charts_def2"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

charts_da_esportare = {
    "01_partecipazione_e_quota.json": grafico_partecipazione,
    "02_medagliere_e_quota.json": grafico_medagliere,
    "03_top15_quota_femminile_paesi.json": grafico_top15,
    "04_correlazioni_indicatori_per_genere.json": grafico_correlazioni,
    "05_scatter_pil_pro_capite_regressione.json": grafico_pil,
    "06_medagliere_principali_paesi_2000_2020.json": grafico_paesi_2000,
}

for filename, chart in charts_da_esportare.items():
    chart.save(CHARTS_DIR / filename)

print(f"Esportati {len(charts_da_esportare)} grafici in: {CHARTS_DIR}")

Esportati 6 grafici in: C:\Users\smacchiavelli\development\code\master\progettone\src\stefano\charts_def2
